In [134]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

## Questão 8 - Sistema de recomendação

**Cenário**

A Marina percebeu que clientes que compram lanchas quase sempre esquecem de levar a defensa (proteção lateral). Ela quer implementar uma vitrine de "Quem comprou isso, também levou..." no site. 

Como não temos ferramentas de Big Data caras, você precisará criar um motor de recomendação, baseado na similaridade de compra dos clientes. 

Identificar qual produto deve ser recomendado junto ao item “GPS Garmin Vortex Maré Drift”, com base na similaridade de comportamento de compra dos clientes.


**Tarefa:**
1. Crie uma matriz de interação Usuário × Produto obedecendo às regras abaixo:
     - a. Linhas: id_cliente
     - b. Colunas: id_produto
     - c. Valor da célula:
     - . 1 se o cliente comprou ao menos uma vez o produto
     - . 0 caso contrário
     - . Ignore a quantidade comprada (presença/ ausência apenas)
2. Cálculo de Similaridade entre Produtos
     - a. Calcule a Similaridade de Cosseno (Cosine Similarity) entre os vetores dos produtos
     - b. A similaridade deve ser calculada produto × produto, com base nos clientes que compraram cada item
3. Ranking de Produtos Similares
     - a. Considere o produto “GPS Garmin Vortex Maré Drift” como item de referência
     - b. Gere um ranking com os 5 produtos mais similares a ele
     - c. Desconsidere o próprio GPS no ranking

In [44]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [41]:
# carregando os dados
df_vendas = pd.read_csv('vendas_corrigidas.csv')
df_produtos = pd.read_csv('produtos_cleaned.csv')

# merge para ter nomes dos produtos
df = df_vendas.merge(df_produtos, left_on='id_product', right_on='code')

# criar coluna binária (presença)
df['comprou'] = 1

# matriz usuário x produto
matriz_user_product = df.pivot_table(
    index='id_client',
    columns='id_product',
    values='comprou',
    aggfunc='max',   # garante 1 mesmo com múltiplas compras
    fill_value=0
)

In [66]:
# Check

matriz_user_product

id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_client,,,,,,,,,,,,,,,,,,,,,
1,1,0,1,1,1,1,0,0,0,0,...,1,1,0,1,1,1,1,1,0,0
2,0,1,1,0,1,1,1,1,1,1,...,1,1,1,1,1,1,0,1,1,1
3,1,1,1,1,1,1,1,0,1,1,...,0,1,1,1,0,1,1,1,1,1
4,1,1,0,1,1,0,1,0,0,1,...,1,1,1,1,0,0,1,0,1,1
5,1,1,0,1,1,1,1,1,1,1,...,1,1,1,1,0,1,1,1,1,0
6,1,0,1,0,1,1,1,0,1,0,...,0,1,1,1,1,0,1,1,1,1
7,1,1,1,0,0,1,1,0,1,1,...,0,1,1,1,1,1,1,1,1,1
8,1,1,1,1,1,1,1,1,1,0,...,1,1,0,1,1,0,1,0,1,1
9,1,1,0,1,1,0,1,1,0,1,...,1,1,1,0,1,1,1,1,1,1


In [67]:
# Transpondo a matriz (produtos como linhas, pois queremos similaridade entre produtos)
matriz_produto = matriz_user_product.T

# Calculando a Similaridade DE cosseno entre todos os pares de produtos
similaridade = cosine_similarity(matriz_produto)

# Transforma em dataframe para facilitar a leitura
df_similaridade = pd.DataFrame(
    similaridade,
    index=matriz_produto.index,
    columns=matriz_produto.index
)

print(df_similaridade.shape)
df_similaridade

(150, 150)


id_product,1,2,3,4,5,6,7,8,9,10,...,141,142,143,144,145,146,147,148,149,150
id_product,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.775058,0.737865,0.810191,0.748331,0.769484,0.775058,0.727825,0.711512,0.698771,...,0.795133,0.753819,0.775058,0.769484,0.721605,0.775058,0.872082,0.721605,0.727825,0.775058
2,0.775058,1.000000,0.704295,0.757865,0.714286,0.712931,0.771429,0.750290,0.788811,0.687256,...,0.795192,0.778078,0.771429,0.767772,0.685714,0.742857,0.712931,0.628571,0.750290,0.742857
3,0.737865,0.704295,1.000000,0.800641,0.704295,0.865181,0.732467,0.712396,0.777778,0.707107,...,0.675923,0.739795,0.732467,0.757033,0.704295,0.788811,0.729996,0.732467,0.739795,0.704295
4,0.810191,0.757865,0.800641,1.000000,0.757865,0.753310,0.757865,0.789747,0.773953,0.735980,...,0.831239,0.789747,0.730798,0.779287,0.730798,0.730798,0.805263,0.757865,0.763422,0.757865
5,0.748331,0.714286,0.704295,0.757865,1.000000,0.795192,0.742857,0.694713,0.760639,0.627495,...,0.795192,0.722501,0.714286,0.685511,0.685714,0.714286,0.767772,0.657143,0.722501,0.714286
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
146,0.775058,0.742857,0.788811,0.730798,0.714286,0.767772,0.714286,0.722501,0.760639,0.627495,...,0.712931,0.722501,0.714286,0.740351,0.714286,1.000000,0.712931,0.771429,0.722501,0.657143
147,0.872082,0.712931,0.729996,0.805263,0.767772,0.763158,0.767772,0.720064,0.702959,0.688247,...,0.789474,0.720064,0.795192,0.763158,0.740351,0.712931,1.000000,0.685511,0.746733,0.795192
148,0.721605,0.628571,0.732467,0.757865,0.657143,0.767772,0.685714,0.750290,0.788811,0.687256,...,0.685511,0.805867,0.714286,0.767772,0.771429,0.771429,0.685511,1.000000,0.750290,0.714286


In [68]:
# Encontrando o id do produto de referência
id_gps = df_produtos[df_produtos['name'] == 'GPS Garmin Vortex Maré Drift']['code'].values[0]

id_gps

27

In [70]:
# Criando o ranking
ranking = df_similaridade[id_gps].sort_values(ascending=False)

# Removendo o próprio produto (id_gps)
ranking = ranking.drop(id_gps)

# Transformando em DataFrame
ranking = ranking.reset_index()

# Dando nomes para as colunas
ranking.columns = ['id_product', 'similaridade']

ranking

,id_product,similaridade
0,94,0.869626
1,11,0.868037
2,35,0.853913
3,115,0.850000
4,1,0.850000
...,...,...
144,60,0.688102
145,120,0.688102
146,43,0.675303
147,75,0.675303


In [71]:
# Apenas os 5 primeiros
ranking_top5 = ranking.head(5)

ranking_top5

,id_product,similaridade
0,94,0.869626
1,11,0.868037
2,35,0.853913
3,115,0.850000
4,1,0.850000


In [72]:
# Trazendo o nome dos produtos

ranking_top5 = ranking_top5.merge(df_produtos[['code', 'name']], left_on='id_product', right_on='code').drop(columns='code')

print(ranking_top5[['id_product', 'name', 'similaridade']])

   id_product                                        name  similaridade
0          94            Motor de Popa Volvo Magnum 276HP      0.869626
1          11         GPS Furuno Swift Leviathan Poseidon      0.868037
2          35                          Radar Furuno Swift      0.853913
3         115  Cabo de Nylon Delta Force Magnum Leviathan      0.850000
4           1                 Transponder AIS Maré Magnum      0.850000
